In [1]:
import kagglehub
!pip install ultralytics
from ultralytics import YOLO
from IPython import display
from IPython.display import display as displayfilechooser
import matplotlib.pyplot as plt
from ipyfilechooser import FileChooser
from pathlib import Path
import pandas as pd
from collections import Counter
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Dataset Downloading and Display

In [2]:
# Download latest version of street signs data set
data_path = kagglehub.dataset_download("pkdarabi/cardetection")

print("data set stored at:" + data_path)
data_yaml = f"{data_path}/car/data.yaml"

100%|██████████| 99.8M/99.8M [00:03<00:00, 29.8MB/s]

Extracting files...


data set stored at:/root/.cache/kagglehub/datasets/pkdarabi/cardetection/versions/5


Dataset Summary:

In [3]:
c = Counter()

# Class IDs from the dataset (based on model validation output)
signs = {
    0: 'Green Light',
    1: 'Red Light',
    2: "Speed Limit 10",
    3: 'Speed Limit 100',
    4: 'Speed Limit 110',
    5: 'Speed Limit 120',
    6: 'Speed Limit 20',
    7: 'Speed Limit 30',
    8: 'Speed Limit 40',
    9: 'Speed Limit 50',
    10: 'Speed Limit 60',
    11: 'Speed Limit 70',
    12: 'Speed Limit 80',
    13: 'Speed Limit 90',
    14: 'Stop',
}

for metadata_file in Path(data_path).rglob("*.txt"):
    content = metadata_file.read_text().strip()
    if content:
        for line in content.splitlines():
            line = line.strip()
            if not line or line.startswith('#'): #some files have a comment
                continue
            try:
                class_id = int(line.split()[0])
                c[class_id] += 1
            except ValueError:
                continue

print("Sign occurrences in dataset:\n")
for class_id, count in sorted(c.items()):
    name = signs.get(class_id, f"Unknown ({class_id})")
    print(f"{name}: {count}")

Sign occurrences in dataset:

Green Light: 774
Red Light: 787
Speed Limit 10: 22
Speed Limit 100: 365
Speed Limit 110: 139
Speed Limit 120: 356
Speed Limit 20: 387
Speed Limit 30: 468
Speed Limit 40: 343
Speed Limit 50: 404
Speed Limit 60: 422
Speed Limit 70: 449
Speed Limit 80: 440
Speed Limit 90: 240
Stop: 416


# Model Training

In [4]:
model = YOLO('yolov8n.pt') #nano size model
data_yaml = f"{data_path}/car/data.yaml"
#just a safety check to prevent accidental time waste
response = input("About to train the model. This can take a very long time. Do you want to proceed? (y/n): ")
if response.lower() == 'y':
    # Train model off dataset
    results = model.train(data=data_yaml, epochs=50, imgsz=416)
else:
    print("Cancelled.")

About to train the model. This can take a very long time. Do you want to proceed? (y/n): y
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/.cache/kagglehub/datasets/pkdarabi/cardetection/versions/5/car/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.93

KeyboardInterrupt: 

# Model Evaluation

In [ ]:
# Evaluate model, generate graphs
metrics = model.val(data=data_yaml, imgsz=416)
print("Model accuracy: ", f"{metrics.box.map50 * 100:.2f}%")
print("Model speed (current hardware): ", f"{sum(metrics.speed.values()):.2f}", "ms")

Training images with bounding boxes and confidence levels.

In [ ]:
display.Image("runs/detect/val/val_batch0_pred.jpg")

Precision-confidence curve compares how accurate the model compared with how confident it was in its output.

In [ ]:
display.Image("runs/detect/val/BoxP_curve.png")

Summary of outputs, line should be as strongly diagonal as possible. Off diagonal outputs are misclassifications.

In [ ]:
display.Image("runs/detect/val/confusion_matrix_normalized.png")

# Upload a file to the model

In [ ]:
class ColabFileUploader:
    def __init__(self):
        self.selected = None
        print("Upload a file:")
        uploaded = files.upload()
        if uploaded:
            self.selected = next(iter(uploaded))
            print(f"File '{self.selected}' uploaded.")

fc = ColabFileUploader()

# Runs the model on the uploaded image

In [ ]:
if fc.selected:
    results = model.predict(fc.selected)

    plt.imshow(results[0].plot(pil=True))
    plt.axis('off')
    plt.show()

else:
    print("No file selected.")